<a href="https://colab.research.google.com/github/natsukosugimura/DeepLearning2/blob/main/2026_python_tutorial_dl_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day2 深層学習による画像分類

## CNNを用いた手書き文字の分類

以下のソースコードは書籍「[TensorFlowとKerasで動かしながら学ぶディープラーニングの仕組み](https://book.mynavi.jp/ec/products/detail/id=109925)」の[ソースコード](https://github.com/enakai00/colab_tfbook)を参考にしています。

In [ ]:
# 基本的なライブラリ
!pip install japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from pandas import DataFrame

# データセットのためのライブラリ
import tensorflow as tf
from tensorflow.keras.datasets import mnist

In [ ]:
# MNISTデータをダウンロードしてNumPyの配列に変換
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape(
                  (len(train_images), 784)).astype('float32') / 255
test_images = test_images.reshape(
                  (len(test_images), 784)).astype('float32') / 255
train_labels = tf.keras.utils.to_categorical(train_labels, 10)
test_labels = tf.keras.utils.to_categorical(test_labels, 10)

In [ ]:
print('学習データの枚数：' + str(len(train_images)))
print('テストデータの枚数：' + str(len(test_images)))

In [ ]:
# 学習データの1枚目のデータを確認
train_images[0]

In [ ]:
# 1枚の画像は28×28ピクセルのデータ
train_images[0].shape

In [ ]:
# 1枚目の画像は"5"を表している
train_labels[0]

In [ ]:
# 先頭10枚の画像を視覚的に確認してみる
fig = plt.figure(figsize=(8, 4))
for c, (image, label) in enumerate(zip(train_images[:10], train_labels[:10])):
    subplot = fig.add_subplot(2, 5, c+1)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.set_title('%d' % np.argmax(label))
    subplot.imshow(image.reshape((28, 28)),
                   vmin=0, vmax=1, cmap=plt.cm.gray_r)

In [ ]:
# ニューラルネットワークを使うためのモジュールをimport
from tensorflow.keras import layers, models, initializers

# 乱数のシードをセット
np.random.seed(20190222)
tf.random.set_seed(20190222)

In [ ]:
# モデルを定義
model = models.Sequential()
model.add(layers.Reshape((28, 28, 1), input_shape=(28*28,), name='reshape'))
model.add(layers.Conv2D(16, (5, 5), padding='same',
                        kernel_initializer=initializers.TruncatedNormal(),
                        use_bias=True, activation='relu',
                        name='conv_filter'))
model.add(layers.MaxPooling2D((2, 2), name='max_pooling'))
model.add(layers.Flatten(name='flatten'))
model.add(layers.Dense(1024, activation='relu',
                       kernel_initializer=initializers.TruncatedNormal(),
                       name='hidden'))
model.add(layers.Dense(10, activation='softmax', name='softmax'))

model.summary()

In [ ]:
# # 別のモデル
# model = models.Sequential([
#     layers.Reshape((28, 28, 1), input_shape=(28*28,)),
#     layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv_filter'),
#     layers.MaxPooling2D((2, 2), name='max_pooling'),
#     layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.4),
#     layers.Dense(10, activation='softmax'),
# ])

In [ ]:
# モデルをコンパイル
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['acc'])

In [ ]:
# モデルを学習させる
history = model.fit(train_images, train_labels,
                    validation_data=(test_images, test_labels),
                    batch_size=512, epochs=10)

In [ ]:
# accuracyとlossの推移を確認
DataFrame({'acc': history.history['acc'],
           'val_acc': history.history['val_acc']}).plot()
DataFrame({'loss': history.history['loss'],
           'val_loss': history.history['val_loss']}).plot()

In [ ]:
# テストデータで予測して精度を確認
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)

print('test_loss: {}'.format(test_loss))
print('test_acc: {}'.format(test_acc))

In [ ]:
# 中間層(畳み込み層とプーリング層)の出力を取り出すモデルを定義
layer_outputs = [model.get_layer('conv_filter').output,
                 model.get_layer('max_pooling').output]
model2 = models.Model(inputs=model.inputs, outputs=layer_outputs)

In [ ]:
# テストデータの先頭9枚に対する中間層の出力
conv_output, pool_output = model2.predict(test_images[:9])
# 学習後の畳み込み層のフィルターの重み
filter_vals = model.get_layer('conv_filter').get_weights()[0]

In [ ]:
# 畳み込み層でフィルタリングした後の画像(畳み込み層の出力)を確認
num_filters = 16
fig = plt.figure(figsize=(10, num_filters+1))
v_max = np.max(conv_output)

for i in range(num_filters):
    subplot = fig.add_subplot(num_filters+1, 10, 10*(i+1)+1)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.imshow(filter_vals[:,:,0,i], cmap=plt.cm.gray_r)

for i in range(9):
    subplot = fig.add_subplot(num_filters+1, 10, i+2)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.set_title('%d' % np.argmax(test_labels[i]))
    subplot.imshow(test_images[i].reshape((28,28)),
                   vmin=0, vmax=1, cmap=plt.cm.gray_r)

    for f in range(num_filters):
        subplot = fig.add_subplot(num_filters+1, 10, 10*(f+1)+i+2)
        subplot.set_xticks([])
        subplot.set_yticks([])
        subplot.imshow(conv_output[i,:,:,f],
                       vmin=0, vmax=v_max, cmap=plt.cm.gray_r)

In [ ]:
# プーリング層を適用した後の画像を確認
num_filters = 16
fig = plt.figure(figsize=(10, num_filters+1))
v_max = np.max(pool_output)

for i in range(num_filters):
    subplot = fig.add_subplot(num_filters+1, 10, 10*(i+1)+1)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.imshow(filter_vals[:,:,0,i],
                   cmap=plt.cm.gray_r, interpolation='nearest')

for i in range(9):
    subplot = fig.add_subplot(num_filters+1, 10, i+2)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.set_title('%d' % np.argmax(test_labels[i]))
    subplot.imshow(test_images[i].reshape((28,28)),
                   vmin=0, vmax=1, cmap=plt.cm.gray_r)

    for f in range(num_filters):
        subplot = fig.add_subplot(num_filters+1, 10, 10*(f+1)+i+2)
        subplot.set_xticks([])
        subplot.set_yticks([])
        subplot.imshow(pool_output[i,:,:,f],
                       vmin=0, vmax=v_max, cmap=plt.cm.gray_r)

In [ ]:
# 正しく分類できなかった画像に対する予測結果を確認
fig = plt.figure(figsize=(12, 10))
c = 0
for (image, label) in zip(test_images, test_labels):
    image
    p_val = model.predict(np.array([image]))
    pred = p_val[0]
    prediction, actual = np.argmax(pred), np.argmax(label)
    if prediction == actual: # 正しく分類できている場合はスキップ->以下は間違えたものに対する処理
        continue
    subplot = fig.add_subplot(5, 4, c*2+1)
    subplot.set_xticks([])
    subplot.set_yticks([])
    subplot.set_title('%d / %d' % (prediction, actual))
    subplot.imshow(image.reshape((28, 28)),
                   vmin=0, vmax=1, cmap=plt.cm.gray_r)
    subplot = fig.add_subplot(5, 4, c*2+2)
    subplot.set_xticks(range(10))
    subplot.set_xlim(-0.5, 9.5)
    subplot.set_ylim(0,1)
    subplot.bar(range(10), pred, align='center', edgecolor='b')
    c += 1
    if c == 10:
        break

 訓練したモデルを用いて，自分で書いた文字の識別をさせてみましょう.

以下のいずれかの方法で手書き文字の画像を用意してください：

1. PowerPointの「描画」機能を使用し、手書き文字をスクリーンショット
2. Paintなどのアプリで手書き  
3. 鉛筆で書いた文字を写真撮影

デモでは1番目の方法で作成した手書き文字画像を使います．

In [ ]:
!pip install ipycanvas ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import io
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

# アップロードされた画像ファイルのバイトから前処理を行い、
# 画像をグレースケール化、28×28 にリサイズ、反転して正規化し、1次元配列（784次元）に変換する関数
def preprocess_image(file_bytes):
    # バイト列から画像を読み込み
    image = Image.open(io.BytesIO(file_bytes))
    # グレースケールに変換
    gray_image = image.convert('L')
    # 28×28 にリサイズ
    resized_image = gray_image.resize((28, 28))
    # NumPy 配列に変換し、画像のピクセル値を反転
    image_array = np.array(resized_image)
    image_array = 255 - image_array
    # 0〜1 の範囲に正規化
    image_array = image_array.astype('float32') / 255.0
    # 1次元（784次元）の配列に変換
    return image_array.flatten()

# ipywidgets のファイルアップロードウィジェットを作成（画像ファイルのみの受け付け）
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)

# 「画像を表示」ボタンがクリックされた場合に、オリジナル画像と前処理画像を表示する関数
def on_show_clicked(b):
    if uploader.value:
        # アップロードされたファイルの最初の要素を取得
        uploaded_file = next(iter(uploader.value.values()))
        file_bytes = uploaded_file['content']
        # オリジナル画像の表示などの処理

        # オリジナル画像の表示
        original_image = Image.open(io.BytesIO(file_bytes))
        plt.figure(figsize=(4, 4))
        plt.imshow(original_image, cmap='gray')
        plt.title('オリジナル画像')
        plt.axis('off')
        plt.show()

        # 前処理画像の表示

        processed = preprocess_image(file_bytes)
        plt.figure(figsize=(4, 4))
        plt.imshow(processed.reshape(28, 28), cmap='gray')
        plt.title('前処理後の画像')
        plt.axis('off')
        plt.show()
    else:
        print("アップロードされた画像がありません。")

# 「画像を表示」ボタンの作成とイベント登録
show_button = widgets.Button(description="画像を表示")
show_button.on_click(on_show_clicked)
display(show_button)

# 「予測する」ボタンがクリックされた場合に、モデルで予測し結果を出力する関数
def on_predict_clicked(b):
    if uploader.value:
        uploaded_file = next(iter(uploader.value.values()))
        file_bytes = uploaded_file['content']
        processed = preprocess_image(file_bytes)
        # 入力形状を (1, 784) にリシェイプしてモデル予測
        result = model.predict(processed.reshape(1, -1))
        predicted_class = np.argmax(result, axis=1)[0]
        print("ニューラルネットワークの予想は:", predicted_class)
    else:
        print("アップロードされた画像がありません。")

# 「予測する」ボタンの作成とイベント登録
predict_button = widgets.Button(description="予測する")
predict_button.on_click(on_predict_clicked)
display(predict_button)